In [2]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl==0.15.2 triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [14]:
!unzip lora_model_32_32-20250530T133353Z-1-001.zip


Archive:  lora_model_32_32-20250530T133353Z-1-001.zip
  inflating: lora_model_32_32/adapter_config.json  
  inflating: lora_model_32_32/README.md  
  inflating: lora_model_32_32/tokenizer_config.json  
  inflating: lora_model_32_32/special_tokens_map.json  
  inflating: lora_model_32_32/chat_template.jinja  
  inflating: lora_model_32_32/tokenizer.model  
  inflating: lora_model_32_32/tokenizer.json  
  inflating: lora_model_32_32/adapter_model.safetensors  


In [1]:
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

In [2]:
if True:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model_32_32", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = max_seq_length,
        dtype = dtype,
        load_in_4bit = load_in_4bit,
    )
    FastLanguageModel.for_inference(model) # Enable native 2x faster inference

alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""
inputs = tokenizer(
[
    alpaca_prompt.format(
        "Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.",  # instruction
        """Question: What is the role of attention mechanisms in transformer models?

          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

          Mark Scheme:
          1. Describes attention as weighing the importance of different words.
          2. Mentions the ability to focus on relevant parts of input.
          3. Explains how attention captures context or relationships.
          4. Refers to handling long-range dependencies or position-independence.""",  # input
        "",  # output - leave this blank for generation!
    )
], return_tensors = "pt").to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer)
_ = model.generate(**inputs, streamer = text_streamer, max_new_tokens = 128)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.5.9: Fast Mistral patching. Transformers: 4.52.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/4.14G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/157 [00:00<?, ?B/s]

Unsloth: Will load lora_model_32_32 as a legacy tokenizer.
Unsloth 2025.5.9 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


<s>Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.

### Input:
Question: What is the role of attention mechanisms in transformer models?

          Reference Answer: Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.

          Student Answer: Attention tells the model which words matter more. It helps with understanding context even when words are far apart.

          Mark Scheme:
          1. Describes attention as weighing the import

In [4]:
!pip install --upgrade gradio


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.2/54.2 MB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.1/323.1 kB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 90.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 7.1 MB/s eta 0:00:00


In [4]:
import gradio as gr
import torch
from unsloth import FastLanguageModel
from transformers import AutoTokenizer

# --- examples ---
examples = [
    {
        "question": "Explain the greenhouse effect.",
        "reference_answer": (
            "The greenhouse effect is the trapping of heat in the Earth's atmosphere by greenhouse gases such as carbon dioxide and methane. "
            "These gases allow sunlight to enter but prevent heat from escaping, leading to warming."
        ),
        "student_answer": (
            "The greenhouse effect happens when the sun’s heat is trapped in the atmosphere by gases like CO2. "
            "This makes the Earth warmer."
        ),
        "mark_scheme": {
            "1": "Defines the greenhouse effect correctly",
            "2": "Mentions greenhouse gases (e.g., CO2, methane)",
            "3": "Explains the mechanism (sunlight in, heat trapped)",
            "4": "Mentions warming or climate impact"
        }
    },
    {
        "question": "What is the role of attention mechanisms in transformer models?",
        "reference_answer": (
            "Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. "
            "They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. "
            "This is critical for handling long-range dependencies in language."
        ),
        "student_answer": (
            "Attention tells the model which words matter more. It helps with understanding context even when words are far apart."
        ),
        "mark_scheme": {
            "1": "Describes attention as weighing the importance of different words.",
            "2": "Mentions the ability to focus on relevant parts of input.",
            "3": "Explains how attention captures context or relationships.",
            "4": "Refers to handling long-range dependencies or position-independence."
        }
    },
]

for i in range(3, 11):
    examples.append({
        "question": f"Example question {i}",
        "reference_answer": f"This is the reference answer for example {i}.",
        "student_answer": f"This is the student's answer for example {i}.",
        "mark_scheme": {
            "1": "Criterion 1",
            "2": "Criterion 2",
            "3": "Criterion 3",
            "4": "Criterion 4"
        }
    })

# --- Format mark scheme dictionary to string ---
def format_mark_scheme(ms_dict):
    return "\n".join(f"{k}: {v}" for k, v in ms_dict.items())

# --- Load example data ---
def load_example_data(choice):
    if choice == "Write your own":
        return "", "", "", ""
    else:
        idx = int(choice.split()[1]) - 1
        ex = examples[idx]
        return (ex["question"], ex["reference_answer"], ex["student_answer"], format_mark_scheme(ex["mark_scheme"]))

# --- Load model and tokenizer ---
model_name = "lora_model_32_32"
tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = FastLanguageModel.for_inference(model_name)
# device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Grading function using your model ---
def grade_essay(question, reference, student, mark_scheme):
    prompt = f"""<s> Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.

### Input:
Question: {question}

Reference Answer: {reference}

Student Answer: {student}

Mark Scheme:
{mark_scheme}

### Response:"""

    inputs = tokenizer([prompt], return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=128)
    result = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return result.strip()

# --- Gradio UI with your design and functionality ---
with gr.Blocks(css="""
    body {background-color: #f9fafb; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;}
    #header {font-weight: 700; font-size: 28px; margin-bottom: 30px; text-align: center; color: #2c3e50;}
    #example-dropdown {margin-bottom: 20px;}
    #grade-button {
        background: #27ae60 !important;
        color: white !important;
        font-weight: 700 !important;
        font-size: 18px !important;
        border-radius: 10px !important;
        padding: 12px 0 !important;
        margin-top: 15px !important;
        box-shadow: 0 5px 15px rgba(39, 174, 96, 0.4);
        transition: background 0.3s, box-shadow 0.3s;
    }
    #grade-button:hover {
        background: #1e8449 !important;
        box-shadow: 0 8px 20px rgba(30, 132, 73, 0.6);
    }
    .gr-textbox {
        border-radius: 12px !important;
        border: 1.5px solid #d1d5db !important;
        padding: 12px !important;
        font-size: 16px !important;
        color: #34495e !important;
        background-color: white !important;
        transition: border-color 0.3s;
    }
    .gr-textbox:focus {
        border-color: #3498db !important;
        box-shadow: 0 0 8px rgba(52, 152, 219, 0.4) !important;
        outline: none !important;
    }
    label {
        font-weight: 600;
        color: #34495e;
        margin-bottom: 6px;
        display: block;
    }
    #output label {
        margin-top: 20px;
        font-size: 18px;
    }
""") as demo:

    gr.Markdown("<div id='header'>📝 Intelligent Essay Grading System</div>")

    example_choices = [f"Example {i+1}" for i in range(len(examples))]
    example_choices.append("Write your own")

    example_dropdown = gr.Dropdown(label="📂 Select Example or Write Your Own", choices=example_choices, value="Write your own", elem_id="example-dropdown")

    with gr.Row():
        with gr.Column(scale=1):
            question_input = gr.Textbox(label="❓ Question", lines=2, placeholder="Enter the essay question here...")
            reference_input = gr.Textbox(label="📖 Reference Answer", lines=6, placeholder="Enter the expert reference answer here...")
            student_input = gr.Textbox(label="📝 Student's Answer", lines=6, placeholder="Enter the student's answer here...")
            mark_scheme_input = gr.Textbox(label="📋 Mark Scheme (one criterion per line, e.g. '1: Criterion description')", lines=6, placeholder="1: Criterion 1\n2: Criterion 2\n3: Criterion 3\n4: Criterion 4")
            grade_button = gr.Button("Grade Essay", elem_id="grade-button")

        with gr.Column(scale=1):
            output = gr.Textbox(label="📊 Grading Result (Score & Rationale)", lines=20, interactive=False, elem_id="output")

    example_dropdown.change(fn=load_example_data, inputs=example_dropdown, outputs=[question_input, reference_input, student_input, mark_scheme_input])

    grade_button.click(fn=grade_essay,
                       inputs=[question_input, reference_input, student_input, mark_scheme_input],
                       outputs=output)

demo.launch()


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c4a8beeb1869078896.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [5]:
import gradio as gr
import torch
from unsloth import FastLanguageModel
from transformers import TextStreamer

# --- Load model using Unsloth ---
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "lora_model_32_32",  # Model directory
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model)  # Enable optimized inference

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- Examples ---
examples = [
    {
        "question": "Explain the greenhouse effect.",
        "reference_answer": "The greenhouse effect is the trapping of heat in the Earth's atmosphere by greenhouse gases such as carbon dioxide and methane. These gases allow sunlight to enter but prevent heat from escaping, leading to warming.",
        "student_answer": "The greenhouse effect happens when the sun’s heat is trapped in the atmosphere by gases like CO2. This makes the Earth warmer.",
        "mark_scheme": {
            "1": "Defines the greenhouse effect correctly",
            "2": "Mentions greenhouse gases (e.g., CO2, methane)",
            "3": "Explains the mechanism (sunlight in, heat trapped)",
            "4": "Mentions warming or climate impact"
        }
    },
    {
        "question": "What is the role of attention mechanisms in transformer models?",
        "reference_answer": "Attention mechanisms allow transformer models to weigh the importance of different words in a sequence when making predictions. They help the model focus on relevant parts of the input, regardless of their position, enabling it to capture context and relationships effectively. This is critical for handling long-range dependencies in language.",
        "student_answer": "Attention tells the model which words matter more. It helps with understanding context even when words are far apart.",
        "mark_scheme": {
            "1": "Describes attention as weighing the importance of different words.",
            "2": "Mentions the ability to focus on relevant parts of input.",
            "3": "Explains how attention captures context or relationships.",
            "4": "Refers to handling long-range dependencies or position-independence."
        }
    },
]

for i in range(3, 11):
    examples.append({
        "question": f"Example question {i}",
        "reference_answer": f"This is the reference answer for example {i}.",
        "student_answer": f"This is the student's answer for example {i}.",
        "mark_scheme": {
            "1": "Criterion 1",
            "2": "Criterion 2",
            "3": "Criterion 3",
            "4": "Criterion 4"
        }
    })

def format_mark_scheme(ms_dict):
    return "\n".join(f"{k}: {v}" for k, v in ms_dict.items())

def load_example_data(choice):
    if choice == "Write your own":
        return "", "", "", ""
    else:
        idx = int(choice.split()[1]) - 1
        ex = examples[idx]
        return (ex["question"], ex["reference_answer"], ex["student_answer"], format_mark_scheme(ex["mark_scheme"]))

# --- Alpaca prompt format ---
def grade_essay(question, reference, student, mark_scheme):
    alpaca_prompt = f"""<s> Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Grade the student's answer to the essay question based on the reference answer and the provided mark scheme. Give a score and rationale.

### Input:
Question: {question}

Reference Answer: {reference}

Student Answer: {student}

Mark Scheme:
{mark_scheme}

### Response:"""

    inputs = tokenizer([alpaca_prompt], return_tensors="pt").to(device)
    outputs = model.generate(**inputs, max_new_tokens=128)
    result = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True)
    return result.strip()

# --- Gradio Interface ---
with gr.Blocks(css="""
    body {background-color: #f9fafb; font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;}
    #header {font-weight: 700; font-size: 28px; margin-bottom: 30px; text-align: center; color: #2c3e50;}
    #example-dropdown {margin-bottom: 20px;}
    #grade-button {
        background: #27ae60 !important;
        color: white !important;
        font-weight: 700 !important;
        font-size: 18px !important;
        border-radius: 10px !important;
        padding: 12px 0 !important;
        margin-top: 15px !important;
        box-shadow: 0 5px 15px rgba(39, 174, 96, 0.4);
        transition: background 0.3s, box-shadow 0.3s;
    }
    #grade-button:hover {
        background: #1e8449 !important;
        box-shadow: 0 8px 20px rgba(30, 132, 73, 0.6);
    }
    .gr-textbox {
        border-radius: 12px !important;
        border: 1.5px solid #d1d5db !important;
        padding: 12px !important;
        font-size: 16px !important;
        color: #34495e !important;
        background-color: white !important;
        transition: border-color 0.3s;
    }
    .gr-textbox:focus {
        border-color: #3498db !important;
        box-shadow: 0 0 8px rgba(52, 152, 219, 0.4) !important;
        outline: none !important;
    }
    label {
        font-weight: 600;
        color: #34495e;
        margin-bottom: 6px;
        display: block;
    }
    #output label {
        margin-top: 20px;
        font-size: 18px;
    }
""") as demo:

    gr.Markdown("<div id='header'>📝 Intelligent Essay Grading System</div>")

    example_choices = [f"Example {i+1}" for i in range(len(examples))] + ["Write your own"]
    example_dropdown = gr.Dropdown(label="📂 Select Example or Write Your Own", choices=example_choices, value="Write your own", elem_id="example-dropdown")

    with gr.Row():
        with gr.Column(scale=1):
            question_input = gr.Textbox(label="❓ Question", lines=2, placeholder="Enter the essay question here...")
            reference_input = gr.Textbox(label="📖 Reference Answer", lines=6, placeholder="Enter the expert reference answer here...")
            student_input = gr.Textbox(label="📝 Student's Answer", lines=6, placeholder="Enter the student's answer here...")
            mark_scheme_input = gr.Textbox(label="📋 Mark Scheme (one criterion per line, e.g. '1: Criterion description')", lines=6, placeholder="1: Criterion 1\n2: Criterion 2\n3: Criterion 3\n4: Criterion 4")
            grade_button = gr.Button("Grade Essay", elem_id="grade-button")

        with gr.Column(scale=1):
            output = gr.Textbox(label="📊 Grading Result (Score & Rationale)", lines=20, interactive=False, elem_id="output")

    example_dropdown.change(fn=load_example_data, inputs=example_dropdown, outputs=[question_input, reference_input, student_input, mark_scheme_input])
    grade_button.click(fn=grade_essay, inputs=[question_input, reference_input, student_input, mark_scheme_input], outputs=output)

demo.launch()


==((====))==  Unsloth 2025.5.9: Fast Mistral patching. Transformers: 4.52.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Unsloth: Will load lora_model_32_32 as a legacy tokenizer.


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://1d20f38b3077ad84f6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
